# Lab 4: Develop a Multi-Agent System

In this lab, we build a modern, production-ready multi-agent system using the latest Azure AI Python SDKs and best practices.
- Each agent is created as a connected agent using Azure AI Agent Service.
- Orchestration is performed using direct agent-to-agent calls, not just a group chat or plugin pattern.
- Uses official Microsoft documentation patterns for agent creation, tool/resource registration, and message passing.


### Part 1: Create the Search, Report, and Validation Agents

#### Step 1: Load packages

In [ ]:
import os
import json
import time
import urllib.request
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import FunctionTool, ToolSet

# Load environment variables
load_dotenv()


#### Step 2: Connect to your Microsoft Foundry Project

In [ ]:
# Use AzureCliCredential (requires az login) for the agents API which needs token-based auth
credential = AzureCliCredential()

# Connecting to our Microsoft Foundry project
project = AIProjectClient(
    endpoint=os.getenv("AIPROJECT_ENDPOINT"),
    credential=credential
)


#### Step 3: Connect to Azure AI Search

In [ ]:
# First enter the name of your search index

index_name="health-plan"
print(index_name)

In [ ]:
# Find Azure AI Search connection and retrieve API key
search_endpoint = None
search_api_key = None

for conn in project.connections.list():
    conn_type = str(getattr(conn, "type", ""))
    if "SEARCH" in conn_type.upper() or "CognitiveSearch" in conn_type:
        conn_full = project.connections.get(name=conn.name, include_credentials=True)
        search_endpoint = conn_full.as_dict().get("target", "").rstrip("/")
        search_api_key = conn_full.as_dict().get("credentials", {}).get("key")
        print(f"Found AI Search connection: {conn.name}")
        break

if not search_endpoint or not search_api_key:
    raise ValueError("Could not retrieve AI Search connection details.")

# Define a Python function that calls Azure AI Search directly with the stored API key
def search_health_plan_index(query: str) -> str:
    """
    Searches the health-plan Azure AI Search index for documents relevant to the given query.
    Returns the top matching document chunks as a JSON string.

    :param query: The search query string.
    :return: JSON string with the top search results (title and chunk).
    """
    url = f"{search_endpoint}/indexes/{index_name}/docs/search?api-version=2023-11-01"
    body = json.dumps({"search": query, "queryType": "simple", "top": 5, "select": "title,chunk"}).encode()
    req = urllib.request.Request(
        url, data=body,
        headers={"api-key": search_api_key, "Content-Type": "application/json"},
        method="POST"
    )
    resp = urllib.request.urlopen(req)
    data = json.loads(resp.read())
    results = [{"title": d.get("title", ""), "chunk": d.get("chunk", "")} for d in data.get("value", [])]
    return json.dumps(results, ensure_ascii=False)

# Wrap in a FunctionTool so the search agent can call it
search_toolset = ToolSet()
search_toolset.add(FunctionTool(functions={search_health_plan_index}))
print("Search tool ready.")


#### Step 4: Create the Search Agent
To create the Search Agent, we use the Azure AI Agent Service SDK to define a dedicated agent that specializes in searching our Azure AI Search index for health plan documents.

In [ ]:
# Search Agent — uses FunctionTool to call Azure AI Search directly
search_agent = project.agents.create_agent(
    model=os.getenv("CHAT_MODEL"),
    name="search-agent",
    instructions="You are a helpful agent that is an expert at searching health plan documents. Use the search_health_plan_index tool to retrieve relevant information, then summarize what you find.",
    toolset=search_toolset
)
print(f"Created search agent, ID: {search_agent.id}")


#### Step 5: Create the Report Agent
Similarly, to create the Report Agent, we use the Azure AI Agent Service SDK to define an agent dedicated to generating detailed reports about health plans. This agent is configured with a specialized system prompt and can be easily orchestrated alongside other agents in the workflow.

In [ ]:
# Report Agent
report_agent = project.agents.create_agent(
    model=os.getenv("CHAT_MODEL"),
    name="report-agent",
    instructions="You are a helpful agent that writes detailed reports about health plans."
)


#### Step 6: Create the Validation Agent
To create the Validation Agent, we again use the Azure AI Agent Service SDK to define an agent focused on validating that generated reports meet specific requirements. The Validation Agent is configured with instructions to check for required content (such as coverage exclusions) and to return a simple pass/fail result. This agent can be invoked programmatically as part of the multi-agent workflow, ensuring that all generated reports adhere to business rules before being delivered to the user.

In [ ]:
# Validation Agent
validation_agent = project.agents.create_agent(
    model=os.getenv("CHAT_MODEL"),
    name="validation-agent",
    instructions="You are a helpful agent that validates reports. Return 'Pass' if the report meets requirements (must include coverage exclusions), otherwise return 'Fail'. Only return 'Pass' or 'Fail'."
)


### Part 2: Orchestrate the Multi-Agent System

Now that we've created our three task agents, the Search, Report, and Validation agents, we can put it all together and create a multi-agent system. We'll use Semantic Kernel to create an Orchestrator Agent that will leverage the three agents to create a report about a health plan.

When you run the below cell, you will see a chat box pop up at the top of VS Code asking you to input the name of a health plan. If you recall, we uploaded two health plans to the search index. Type one of the following health plans in the box and press enter to begin running the multi-agent system:

- Northwind Health Standard
- Northwind Health Plus

The orchestration code below...
- Defines an `orchestrate` function to coordinate the multi-agent workflow for a given health plan name:
  - The Search Agent retrieves information about the specified health plan from Azure AI Search.
  - The Report Agent generates a detailed report using the information returned by the Search Agent.
  - The Validation Agent checks that the report includes required content (coverage exclusions) and returns 'Pass' or 'Fail'.
  - If validation passes, the report is saved to a markdown file; otherwise, a message is printed indicating the report did not meet requirements.
- Defines a helper function to extract the last agent/assistant message from a list of messages.
- Provides a command-line interface to interactively enter health plan names, generate reports, and exit the system.
- Cleans up by deleting all agents when finished.

In [ ]:
def extract_last_agent_message(messages):
    """Extract text from the last agent/assistant message, handling MessageTextContent objects."""
    last_msg = None
    for msg in reversed(list(messages)):
        role = str(getattr(msg, "role", ""))
        if "agent" in role.lower() or "assistant" in role.lower():
            last_msg = msg
            break
    if not last_msg or not isinstance(getattr(last_msg, "content", None), list):
        return ""
    for part in last_msg.content:
        # Handle MessageTextContent SDK objects
        if hasattr(part, "text"):
            text = part.text
            val = getattr(text, "value", None) or (text.get("value") if isinstance(text, dict) else None)
            if val:
                return val
        # Handle plain dicts
        elif isinstance(part, dict) and part.get("type") == "text":
            val = part.get("text", {}).get("value")
            if val:
                return val
    return ""


def run_search_agent(user_content: str) -> str:
    """Run the search agent with manual FunctionTool polling and return the response text."""
    thread = project.agents.threads.create()
    project.agents.messages.create(thread_id=thread.id, role="user", content=user_content)
    run = project.agents.runs.create(thread_id=thread.id, agent_id=search_agent.id)

    while run.status in ("queued", "in_progress", "requires_action"):
        time.sleep(1)
        run = project.agents.runs.get(thread_id=thread.id, run_id=run.id)
        if run.status == "requires_action":
            tool_outputs = []
            for tc in run.required_action.submit_tool_outputs.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                print(f"  Search agent calling: {fn_name}({list(fn_args.keys())})")
                result = search_health_plan_index(**fn_args) if fn_name == "search_health_plan_index" else json.dumps({"error": f"Unknown: {fn_name}"})
                tool_outputs.append({"tool_call_id": tc.id, "output": result})
            project.agents.runs.submit_tool_outputs(thread_id=thread.id, run_id=run.id, tool_outputs=tool_outputs)

    if run.status == "failed":
        raise RuntimeError(f"Search agent run failed: {run.last_error}")
    return extract_last_agent_message(project.agents.messages.list(thread_id=thread.id))


def run_simple_agent(agent_id: str, user_content: str) -> str:
    """Run a no-tool agent (report or validation) and return the response text."""
    thread = project.agents.threads.create()
    project.agents.messages.create(thread_id=thread.id, role="user", content=user_content)
    run = project.agents.runs.create_and_process(thread_id=thread.id, agent_id=agent_id)
    if run.status == "failed":
        raise RuntimeError(f"Agent run failed: {run.last_error}")
    return extract_last_agent_message(project.agents.messages.list(thread_id=thread.id))


def orchestrate(plan_name: str):
    print(f"\n[1/3] Search agent retrieving info about '{plan_name}'...")
    plan_info = run_search_agent(f"Tell me about the {plan_name} plan.")

    print(f"[2/3] Report agent writing the report...")
    report_content = run_simple_agent(
        report_agent.id,
        f"Write a detailed report about the {plan_name} plan. Include coverage exclusions. Here is the relevant information: {plan_info}"
    )

    print(f"[3/3] Validation agent checking the report...")
    validation_result = run_simple_agent(
        validation_agent.id,
        f"Validate that the following report includes coverage exclusions. Here is the report: {report_content}"
    )

    if validation_result.strip().lower() == "pass":
        filename = f"{plan_name} Report.md"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"Report validated and saved to: {filename}")
        return {"report_was_generated": True, "content": report_content}
    else:
        print(f"Validation result: {validation_result!r} — report did not meet requirements.")
        return {"report_was_generated": False, "content": "The report could not be generated as it did not meet the required validation standards."}


# Interactive loop — enter a health plan name when prompted
# Available plans: 'Northwind Standard' or 'Northwind Health Plus'
print("Welcome to the Health Plan Multi-Agent System!")
print("Available plans: 'Northwind Standard', 'Northwind Health Plus'")
while True:
    plan_name = input("Enter a health plan name (or 'exit' to quit): ").strip()
    if not plan_name or plan_name.lower() == "exit":
        break
    result = orchestrate(plan_name)
    print(json.dumps({"report_was_generated": result["report_was_generated"]}, indent=2))

# Cleanup
project.agents.delete_agent(search_agent.id)
project.agents.delete_agent(report_agent.id)
project.agents.delete_agent(validation_agent.id)
print("\nAll agents deleted. Goodbye!")
